In [31]:
with open("data/text8.txt", "r", encoding="utf-8") as f:
    text = f.read()

split = int(0.1 * len(text))

# train_text = text[:split]
text = text[:split]
wordList = text.lower().split()
len(wordList)

12107623

In [32]:
import torch

vocab = sorted(set(wordList))

word_to_key =  {word: idx for idx,word in enumerate(vocab)}

tokens = torch.tensor( [word_to_key[word] for word in wordList], dtype=torch.long )

vocab_size = len(vocab)
len(tokens), vocab_size

(12107623, 209174)

In [33]:
t = 1e-5

counts = torch.bincount(
    tokens,
    minlength=vocab_size
).float()

frequencies = counts / counts.sum()

keep_probs = (
    torch.sqrt(t / frequencies)
    + t / frequencies
)

keep_probs = torch.clamp(
    keep_probs,
    max=1.0
)

# Randomly decide which tokens to keep
random_values = torch.rand(len(tokens))

mask = random_values < keep_probs[tokens]

tokens = tokens[mask]

len(tokens)

4188195

In [34]:

windowSize = 2

centers = []
contexts = []

for offset in range(1,windowSize + 1):
    # word[i] -> word[i + offset]
    centers.append(tokens[:-offset])
    contexts.append(tokens[offset:])

    # word[i + offset] -> word[i]
    centers.append(tokens[offset:])
    contexts.append(tokens[:-offset])

centers = torch.cat(centers)
contexts = torch.cat(contexts)

len(centers)

16752774

In [36]:

from torch.utils.data import TensorDataset, DataLoader

text8Dataset = TensorDataset(centers,contexts)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train_loader = DataLoader(
    text8Dataset,
    batch_size=1024,
    shuffle=True,
    pin_memory=(device.type == "cuda"),
    num_workers=0
)

In [37]:
# P(w) ∝ count(w)^0.75
counts = torch.bincount(tokens, minlength=vocab_size ).float()
negative_probs = counts.pow(0.75)

negative_probs /= negative_probs.sum()

# Move once to GPU rather than every batch negative_probs = negative_probs.to(device)
negative_probs = negative_probs.to(device)

In [38]:

import torch.nn as nn
import torch.nn.functional as F


class SkipGram(nn.Module):
    def __init__(self, vocab_size, embeddingDim):
        super().__init__()

        self.W_in = nn.Embedding(vocab_size,embeddingDim)
        self.W_out = nn.Embedding(vocab_size,embeddingDim)

    def forward(self, center, context, negatives):
        v_center =  self.W_in(center)
        v_context =  self.W_out(context)

        pos_score = torch.sum(v_center * v_context, dim=1)
        pos_loss = -F.logsigmoid(pos_score)

        v_neg =  self.W_out(negatives)

        neg_scores = torch.sum(
                v_neg * v_center.unsqueeze(1),
                dim=2
            )

        neg_loss = -F.logsigmoid(-neg_scores).sum(dim=1)

        return (pos_loss + neg_loss).mean()


In [39]:
embeddingDim = 100

model = SkipGram(vocab_size, embeddingDim=embeddingDim).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=0.01
)

In [40]:
num_neg = 5

for epoch in range(5):
    model.train()
    total_loss = 0

    for center,context in train_loader:

        center = center.to(device, non_blocking=True)
        context = context.to(device, non_blocking=True)

        negatives = torch.multinomial(
            negative_probs, 
            center.size(0) * num_neg, 
            replacement=True 
        ).view( 
            center.size(0),
            num_neg
        )

        optimizer.zero_grad(set_to_none=True)

        loss = model(center, context, negatives)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    print(f"Epoch {epoch}: loss = {avg_loss:.4f}")

Epoch 0: loss = 24.1829
Epoch 1: loss = 24.0326
Epoch 2: loss = 23.9030
Epoch 3: loss = 23.7574
Epoch 4: loss = 23.6219


In [41]:
pretrained_embeddings = model.W_in.weight.detach().cpu()

torch.save(
    {
        "embeddings": pretrained_embeddings,
        "word_to_id": word_to_key,
        "vocab": vocab,
    },
    "text8_embeddings.pt"
)